# 🤖 SVM Expense Categorizer — Training Notebook
### Intelligent Expense Tracker — Final Year Project

This notebook:
1. Generates 1000+ synthetic Indian receipt/transaction records
2. Trains Naive Bayes, SVM, and Random Forest models
3. Evaluates and compares all models
4. Saves the best model (SVM) + TF-IDF vectorizer as `.pkl` files
5. Downloads files to your computer for use in Flask backend

**Run each cell top to bottom. Takes ~2 minutes total.**

## Cell 1 — Install & Import Libraries

In [ ]:
# Install required libraries
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

print('✅ All libraries imported successfully!')

## Cell 2 — Generate Synthetic Indian Receipt Data

In [ ]:
# ─────────────────────────────────────────────────────────────
# SYNTHETIC DATA GENERATION
# Each category has realistic Indian vendor names + keywords
# ─────────────────────────────────────────────────────────────

CATEGORY_DATA = {
    'Food': {
        'vendors': [
            'Zomato Order', 'Swiggy Delivery', 'McDonald\'s India',
            'Domino\'s Pizza', 'KFC India', 'Burger King', 'Pizza Hut',
            'Haldiram\'s', 'Bikanervala', 'Cafe Coffee Day', 'Starbucks India',
            'Udupi Hotel', 'Saravana Bhavan', 'Hotel Annapurna',
            'Punjabi Dhaba', 'Biryani House', 'Subway India', 'Wow Momo',
            'Faasos', 'Box8', 'EatFit', 'Chaayos', 'Theobroma',
            'local restaurant', 'canteen', 'food court', 'tiffin center'
        ],
        'keywords': [
            'restaurant meal biryani', 'pizza delivery order', 'burger combo meal',
            'coffee cappuccino latte', 'lunch dinner breakfast thali',
            'food delivery online order', 'snacks beverages cold drink',
            'dosa idli sambar', 'paratha curry dal', 'chinese noodles fried rice',
            'cake pastry dessert', 'ice cream kulfi', 'juice smoothie shake'
        ],
        'amount_range': (80, 800)
    },
    'Travel': {
        'vendors': [
            'Uber India', 'Ola Cabs', 'Rapido Bike', 'IndiGo Airlines',
            'Air India', 'SpiceJet', 'IRCTC Train Ticket', 'RedBus',
            'Abhibus', 'MakeMyTrip', 'Goibibo', 'Yatra.com',
            'BMTC Bus Pass', 'Metro Card Recharge', 'Petrol Pump HPCL',
            'Indian Oil Petrol', 'Bharat Petroleum', 'Auto Rickshaw',
            'City Taxi', 'OYO Rooms', 'Treebo Hotels', 'FabHotels'
        ],
        'keywords': [
            'cab ride airport pickup', 'fuel petrol diesel refill',
            'train ticket reservation irctc', 'flight boarding pass airlines',
            'bus ticket intercity travel', 'hotel room booking night stay',
            'metro card recharge transit', 'auto rickshaw fare trip',
            'toll highway expressway', 'parking charges vehicle',
            'bike rental scooter', 'travel insurance trip'
        ],
        'amount_range': (50, 5000)
    },
    'Groceries': {
        'vendors': [
            'BigBasket', 'Blinkit', 'Zepto', 'JioMart', 'DMart',
            'Reliance Fresh', 'More Supermarket', 'Spencer\'s Retail',
            'Nature\'s Basket', 'Nilgiris', 'Spar Hypermarket',
            'local kirana store', 'vegetable market', 'fruit stall',
            'Hypercity', 'Star Bazaar', 'Grofers'
        ],
        'keywords': [
            'vegetables fruits grocery monthly', 'rice wheat flour dal pulses',
            'milk eggs bread dairy products', 'oil ghee butter cooking essentials',
            'supermarket weekly shopping', 'masala spices condiments',
            'packaged food snacks biscuits', 'detergent soap cleaning household',
            'kirana store daily needs', 'atta maida sugar salt'
        ],
        'amount_range': (200, 3000)
    },
    'Shopping': {
        'vendors': [
            'Amazon India', 'Flipkart', 'Myntra', 'Ajio', 'Nykaa',
            'Meesho', 'Snapdeal', 'Tata CLiQ', 'Shopsy',
            'Pantaloons', 'Westside', 'H&M India', 'Zara India',
            'Lifestyle Stores', 'Central Mall', 'Phoenix Mall',
            'Croma Electronics', 'Vijay Sales', 'Reliance Digital',
            'Lenskart', 'Boat Accessories', 'Decathlon India'
        ],
        'keywords': [
            'clothes dress shirt kurta purchase', 'shoes footwear sneakers sandals',
            'mobile phone accessories gadget', 'electronics laptop headphones',
            'online shopping order delivered', 'fashion clothing apparel',
            'bag wallet purse accessories', 'home decor furniture appliance',
            'sports equipment gym gear', 'books stationery office supplies'
        ],
        'amount_range': (300, 8000)
    },
    'Health': {
        'vendors': [
            'Apollo Pharmacy', 'MedPlus', '1mg Online Pharmacy', 'Netmeds',
            'PharmEasy', 'Apollo Hospitals', 'Fortis Healthcare',
            'Max Hospital', 'Manipal Hospitals', 'AIIMS',
            'local clinic', 'diagnostic lab', 'dental clinic',
            'eye care center', 'Lal Path Labs', 'Dr Lal PathLabs',
            'SRL Diagnostics', 'thyrocare', 'pathology lab'
        ],
        'keywords': [
            'medicine tablets capsules prescription', 'doctor consultation fees',
            'hospital bill medical checkup', 'pharmacy drugs healthcare',
            'blood test diagnostic pathology', 'dental teeth checkup cleaning',
            'eye test spectacles contact lens', 'physiotherapy gym fitness',
            'health insurance premium', 'vitamin supplement protein'
        ],
        'amount_range': (100, 5000)
    },
    'Utilities': {
        'vendors': [
            'BESCOM Electricity', 'MSEDCL Power Bill', 'Tata Power',
            'Airtel Broadband', 'Jio Fiber', 'BSNL Broadband',
            'Vodafone Idea', 'Airtel Mobile Recharge', 'Jio Recharge',
            'Indane Gas Cylinder', 'HP Gas Booking', 'Bharat Gas',
            'BWSSB Water Bill', 'CGWA Water', 'municipal water tax',
            'property tax', 'society maintenance', 'Netflix subscription',
            'Amazon Prime', 'Hotstar Disney+'
        ],
        'keywords': [
            'electricity bill payment monthly units', 'internet broadband wifi recharge',
            'mobile phone recharge prepaid postpaid', 'gas cylinder lpg booking',
            'water bill municipal payment', 'DTH cable tv subscription',
            'streaming ott platform subscription', 'society maintenance charges',
            'property rent house payment', 'insurance premium annual'
        ],
        'amount_range': (100, 2500)
    },
    'Entertainment': {
        'vendors': [
            'BookMyShow', 'PVR Cinemas', 'INOX Movies', 'Cinepolis',
            'Netflix India', 'Spotify Premium', 'YouTube Premium',
            'Steam Gaming', 'PlayStation Store', 'EsportsXO',
            'Imagica Theme Park', 'Wonderla', 'escape room',
            'bowling alley', 'gaming zone', 'concert tickets'
        ],
        'keywords': [
            'movie ticket cinema hall show', 'concert event show ticket booking',
            'gaming subscription console game', 'music streaming spotify apple',
            'theme park amusement rides', 'bowling sports activity',
            'ott platform web series binge', 'night club pub bar',
            'comedy show standup event', 'sports match stadium ticket'
        ],
        'amount_range': (100, 3000)
    },
    'Other': {
        'vendors': [
            'ATM Withdrawal', 'bank charges fee', 'post office',
            'courier service', 'gift purchase', 'donation charity',
            'school fees', 'college tuition', 'coaching classes',
            'salon haircut', 'parlour spa', 'laundry dry cleaning',
            'repair service', 'plumber electrician', 'stationery shop'
        ],
        'keywords': [
            'miscellaneous personal expense', 'gift present birthday',
            'education tuition fees course', 'salon haircut grooming',
            'repair maintenance service', 'courier delivery shipping',
            'charity donation contribution', 'bank charges penalty fee',
            'laundry cleaning service', 'newspaper magazine subscription'
        ],
        'amount_range': (50, 2000)
    }
}

print(f'✅ Category data defined for {len(CATEGORY_DATA)} categories')
print('Categories:', list(CATEGORY_DATA.keys()))

In [ ]:
# ─────────────────────────────────────────────────────────────
# BUILD THE DATASET
# Generate 1200 synthetic transactions (150 per category)
# ─────────────────────────────────────────────────────────────

def generate_dataset(samples_per_category=150):
    records = []
    start_date = datetime(2024, 1, 1)

    for category, data in CATEGORY_DATA.items():
        for i in range(samples_per_category):
            vendor = random.choice(data['vendors'])
            keyword_phrase = random.choice(data['keywords'])

            # Combine vendor + keywords to simulate real OCR text
            combined_text = f"{vendor} {keyword_phrase}"

            # Add slight noise — some records only have vendor name
            if random.random() < 0.2:
                combined_text = vendor

            # Random amount in category range
            lo, hi = data['amount_range']
            amount = round(random.uniform(lo, hi), 2)

            # Random date in 2024
            days_offset = random.randint(0, 364)
            txn_date = start_date + timedelta(days=days_offset)

            records.append({
                'text': combined_text.lower().strip(),
                'vendor_name': vendor,
                'category': category,
                'amount': amount,
                'date': txn_date.strftime('%Y-%m-%d')
            })

    df = pd.DataFrame(records)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle
    return df


df = generate_dataset(samples_per_category=150)

print(f'✅ Dataset generated: {len(df)} total records')
print(f'\n📊 Records per category:')
print(df['category'].value_counts())
print(f'\n🔍 Sample records:')
df.head(8)

## Cell 3 — Visualize the Dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold')

# Plot 1: Category distribution
colors = ['#f97316','#3b82f6','#22c55e','#a855f7','#ec4899','#eab308','#06b6d4','#6b7280']
cat_counts = df['category'].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, color=colors)
axes[0].set_title('Records per Category')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Average amount per category
avg_amounts = df.groupby('category')['amount'].mean().reindex(cat_counts.index)
axes[1].bar(avg_amounts.index, avg_amounts.values, color=colors)
axes[1].set_title('Average Amount per Category (₹)')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Average Amount (₹)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dataset visualization saved!')

## Cell 4 — Preprocess Text & Split Data

In [ ]:
import re
from sklearn.preprocessing import LabelEncoder

def clean_text(text):
    """Clean and normalize transaction text."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)   # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()    # normalize spaces
    # Remove common stopwords manually
    stopwords = {'the','a','an','and','or','in','on','at','to','for','of','with','by','from','is','was'}
    words = [w for w in text.split() if w not in stopwords and len(w) > 1]
    return ' '.join(words)

# Clean text
df['clean_text'] = df['text'].apply(clean_text)

# Features and labels
X = df['clean_text']
y = df['category']

# Label encode categories
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Classes:', le.classes_)

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'\n✅ Data split complete:')
print(f'   Training samples : {len(X_train)}')
print(f'   Testing samples  : {len(X_test)}')

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),    # unigrams + bigrams
    min_df=1,
    sublinear_tf=True      # log scaling
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'   TF-IDF features  : {X_train_tfidf.shape[1]}')
print('✅ TF-IDF vectorization done!')

## Cell 5 — Train All 3 Models & Compare

In [ ]:
from time import time

results = {}

# ── Model 1: Naive Bayes ──────────────────────────────────
print('🔵 Training Naive Bayes...')
t0 = time()
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)
nb_preds = nb.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)
nb_f1  = f1_score(y_test, nb_preds, average='macro')
results['Naive Bayes'] = {'accuracy': nb_acc, 'f1': nb_f1, 'time': time()-t0, 'model': nb}
print(f'   Accuracy: {nb_acc:.4f} | F1: {nb_f1:.4f} | Time: {time()-t0:.2f}s')

# ── Model 2: SVM (Linear) ────────────────────────────────
print('\n🟢 Training SVM (LinearSVC)...')
t0 = time()
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm.fit(X_train_tfidf, y_train)
svm_preds = svm.predict(X_test_tfidf)
svm_acc = accuracy_score(y_test, svm_preds)
svm_f1  = f1_score(y_test, svm_preds, average='macro')
results['SVM'] = {'accuracy': svm_acc, 'f1': svm_f1, 'time': time()-t0, 'model': svm}
print(f'   Accuracy: {svm_acc:.4f} | F1: {svm_f1:.4f} | Time: {time()-t0:.2f}s')

# ── Model 3: Random Forest ────────────────────────────────
print('\n🟡 Training Random Forest (100 trees)...')
t0 = time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
rf_preds = rf.predict(X_test_tfidf)
rf_acc = accuracy_score(y_test, rf_preds)
rf_f1  = f1_score(y_test, rf_preds, average='macro')
results['Random Forest'] = {'accuracy': rf_acc, 'f1': rf_f1, 'time': time()-t0, 'model': rf}
print(f'   Accuracy: {rf_acc:.4f} | F1: {rf_f1:.4f} | Time: {time()-t0:.2f}s')

# ── Summary ───────────────────────────────────────────────
print('\n' + '='*55)
print(f'{"Model":<20} {"Accuracy":>10} {"F1-Score":>10} {"Time":>10}')
print('='*55)
for name, r in results.items():
    print(f'{name:<20} {r["accuracy"]:>10.4f} {r["f1"]:>10.4f} {r["time"]:>9.2f}s')
print('='*55)

best_name = max(results, key=lambda k: results[k]['f1'])
print(f'\n🏆 Best Model: {best_name} (F1 = {results[best_name]["f1"]:.4f})')

## Cell 6 — Detailed SVM Evaluation

In [ ]:
# Full classification report for SVM
print('📋 SVM — Detailed Classification Report:')
print('='*60)
print(classification_report(
    y_test, svm_preds,
    target_names=le.classes_
))

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, svm_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True, fmt='d',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    cmap='Blues',
    linewidths=0.5
)
plt.title('SVM Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('svm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved!')

In [ ]:
# Model Comparison Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Model Comparison', fontsize=14, fontweight='bold')

model_names = list(results.keys())
accuracies  = [results[m]['accuracy'] for m in model_names]
f1_scores   = [results[m]['f1'] for m in model_names]
bar_colors  = ['#3b82f6', '#22c55e', '#f59e0b']

axes[0].bar(model_names, accuracies, color=bar_colors)
axes[0].set_title('Accuracy')
axes[0].set_ylim(0.5, 1.0)
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(model_names, f1_scores, color=bar_colors)
axes[1].set_title('F1-Score (Macro)')
axes[1].set_ylim(0.5, 1.0)
for i, v in enumerate(f1_scores):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model comparison chart saved!')

## Cell 7 — Save Models as .pkl Files

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# Save SVM model
joblib.dump(svm,  'models/svm_model.pkl')
print('✅ Saved: models/svm_model.pkl')

# Save TF-IDF vectorizer
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
print('✅ Saved: models/tfidf_vectorizer.pkl')

# Save Label Encoder (so Flask knows category names)
joblib.dump(le, 'models/label_encoder.pkl')
print('✅ Saved: models/label_encoder.pkl')

# Save training dataset as CSV (for reference)
df.to_csv('models/training_data.csv', index=False)
print('✅ Saved: models/training_data.csv')

print(f'\n📦 All model files saved in /models folder')

## Cell 8 — Test the Saved Model (Simulate Flask behavior)

In [ ]:
# Simulate what Flask categorizer.py will do
loaded_svm    = joblib.load('models/svm_model.pkl')
loaded_tfidf  = joblib.load('models/tfidf_vectorizer.pkl')
loaded_le     = joblib.load('models/label_encoder.pkl')

def predict_category(vendor_name, raw_text=''):
    combined = clean_text(f"{vendor_name} {raw_text}")
    vec = loaded_tfidf.transform([combined])
    pred_encoded = loaded_svm.predict(vec)[0]
    return loaded_le.inverse_transform([pred_encoded])[0]

# Test with real-world examples
test_cases = [
    ('Zomato Order', 'biryani delivery food'),
    ('Uber India', 'cab ride airport'),
    ('BigBasket', 'vegetables fruits monthly grocery'),
    ('Apollo Pharmacy', 'medicine tablets prescription'),
    ('Amazon India', 'electronics mobile phone purchase'),
    ('BESCOM Electricity', 'power bill payment monthly'),
    ('BookMyShow', 'movie ticket cinema'),
    ('IRCTC Train Ticket', 'train reservation travel'),
    ('Cafe Coffee Day', 'coffee cappuccino snack'),
    ('local salon', 'haircut grooming personal'),
]

print(f'{"Vendor":<30} {"OCR Text":<35} {"Predicted Category"}')
print('─' * 85)
for vendor, ocr_text in test_cases:
    pred = predict_category(vendor, ocr_text)
    print(f'{vendor:<30} {ocr_text:<35} 🏷️  {pred}')

## Cell 9 — Download Files to Your Computer

In [ ]:
# Zip all model files together for easy download
import shutil
shutil.make_archive('expense_svm_models', 'zip', 'models')
print('✅ Zipped all model files → expense_svm_models.zip')

# Download the zip file
from google.colab import files
files.download('expense_svm_models.zip')
files.download('svm_confusion_matrix.png')
files.download('model_comparison.png')

print('\n📥 Downloads started!')
print('\n📌 NEXT STEPS:')
print('   1. Extract expense_svm_models.zip')
print('   2. Copy svm_model.pkl, tfidf_vectorizer.pkl, label_encoder.pkl')
print('   3. Paste them into your Flask backend/models/ folder')
print('   4. Your categorizer.py will load these automatically')